# Project 01 — Data cleaning

## First-Time Buyer Affordability Pressure by Area

This notebook turns the ONS affordability workbook into an analysis-ready area-year dataset.

The first profiling stage confirmed that the workbook contains multiple tables across separate sheets. For the first-time buyer angle, the cleaning process focuses on local-authority lower-quartile measures: lower-quartile house price, lower-quartile workplace-based earnings and the lower-quartile affordability ratio.


## Cleaning approach

The workbook uses one sheet per measure and stores years across columns. This notebook reshapes the selected sheets into a long area-year format, merges the measures, applies basic data quality checks and saves cleaned outputs for SQL, Python, Excel and Power BI.


In [1]:
from pathlib import Path
from datetime import date
import re

import numpy as np
import pandas as pd


In [2]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
DICTIONARY_DIR = PROJECT_ROOT / "data" / "dictionary"
REPORTS_DIR = PROJECT_ROOT / "reports"

RAW_FILE = RAW_DIR / "aff1ratioofhousepricetoworkplacebasedearnings.xlsx"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
DICTIONARY_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(
        f"Raw workbook not found: {RAW_FILE}. Run notebook 01 first."
    )

print(f"Using raw workbook: {RAW_FILE.relative_to(PROJECT_ROOT)}")


Using raw workbook: data\raw\aff1ratioofhousepricetoworkplacebasedearnings.xlsx


## Selected sheets

The workbook contents show that the local-authority lower-quartile tables are held on sheets 6a, 6b and 6c. The median affordability ratio is also loaded as a comparison measure.


In [3]:
SHEET_CONFIG = {
    "6a": "lower_quartile_house_price",
    "6b": "lower_quartile_annual_earnings",
    "6c": "lower_quartile_affordability_ratio",
    "5c": "median_affordability_ratio",
}

ID_COLUMNS = [
    "country_region_code",
    "country_region_name",
    "local_authority_code",
    "local_authority_name",
]


In [4]:
def extract_year(column):
    """Extract a four-digit year from an Excel column label."""
    match = re.search(r"(19|20)\d{2}", str(column))
    return int(match.group(0)) if match else None


def clean_numeric_series(series):
    """Convert ONS disclosure markers and numeric-looking values into numbers."""
    text_series = series.astype("string").str.strip()
    text_series = text_series.mask(
        text_series.isin(["[x]", "x", ":", "nan", "None", "<NA>"]),
        np.nan
    )
    text_series = text_series.str.replace(",", "", regex=False)
    return pd.to_numeric(text_series, errors="coerce")


def read_measure_sheet(sheet_name, measure_name):
    raw = pd.read_excel(RAW_FILE, sheet_name=sheet_name, header=1)
    raw = raw.dropna(how="all").dropna(axis=1, how="all")

    raw_columns = list(raw.columns)

    rename_map = {
        raw_columns[0]: "country_region_code",
        raw_columns[1]: "country_region_name",
        raw_columns[2]: "local_authority_code",
        raw_columns[3]: "local_authority_name",
    }

    raw = raw.rename(columns=rename_map)

    year_columns = [
        column for column in raw.columns
        if extract_year(column) is not None
    ]

    year_rename_map = {
        column: extract_year(column)
        for column in year_columns
    }

    raw = raw.rename(columns=year_rename_map)
    year_columns = list(year_rename_map.values())

    if not year_columns:
        raise ValueError(
            f"No year columns found in sheet {sheet_name}. Check the header row."
        )

    long = raw[ID_COLUMNS + year_columns].melt(
        id_vars=ID_COLUMNS,
        value_vars=year_columns,
        var_name="year",
        value_name=measure_name,
    )

    long["year"] = long["year"].astype(int)
    long[measure_name] = clean_numeric_series(long[measure_name])

    long = long.dropna(
        subset=[
            "local_authority_code",
            "local_authority_name",
            "year",
        ]
    )

    if long.empty:
        raise ValueError(
            f"Sheet {sheet_name} produced 0 rows after cleaning. Review workbook structure."
        )

    long["source_sheet"] = sheet_name

    return long

In [5]:
measure_frames = []

for sheet_name, measure_name in SHEET_CONFIG.items():
    measure_frame = read_measure_sheet(sheet_name, measure_name)
    print(f"Loaded sheet {sheet_name}: {measure_name} — {measure_frame.shape[0]:,} rows")
    measure_frames.append(measure_frame.drop(columns="source_sheet"))


Loaded sheet 6a: lower_quartile_house_price — 9,222 rows
Loaded sheet 6b: lower_quartile_annual_earnings — 9,222 rows
Loaded sheet 6c: lower_quartile_affordability_ratio — 9,222 rows
Loaded sheet 5c: median_affordability_ratio — 9,222 rows


## Merge measures into one area-year table


In [6]:
cleaned = measure_frames[0]

for frame in measure_frames[1:]:
    cleaned = cleaned.merge(
        frame,
        on=ID_COLUMNS + ["year"],
        how="outer",
        validate="one_to_one",
    )

cleaned = cleaned.sort_values(["local_authority_name", "year"]).reset_index(drop=True)

print(f"Cleaned rows: {cleaned.shape[0]:,}")
print(f"Cleaned columns: {cleaned.shape[1]:,}")
cleaned.head()


Cleaned rows: 9,222
Cleaned columns: 9


,country_region_code,country_region_name,local_authority_code,local_authority_name,year,lower_quartile_house_price,lower_quartile_annual_earnings,lower_quartile_affordability_ratio,median_affordability_ratio
0,E12000008,South East,E07000223,Adur,1997,49950,11662,4.28,3.54
1,E12000008,South East,E07000223,Adur,1998,56500,13091,4.32,3.68
2,E12000008,South East,E07000223,Adur,1999,63000,14091,4.47,3.87
3,E12000008,South East,E07000223,Adur,2000,74375,14496,5.13,4.69
4,E12000008,South East,E07000223,Adur,2001,86000,11774,7.3,6.63


## Data quality checks


In [7]:
duplicate_key_count = cleaned.duplicated(subset=["local_authority_code", "year"]).sum()
missing_key_counts = cleaned[["local_authority_code", "local_authority_name", "year"]].isna().sum()

quality_summary = pd.DataFrame([
    {"check": "row_count", "result": cleaned.shape[0]},
    {"check": "column_count", "result": cleaned.shape[1]},
    {"check": "duplicate_local_authority_year_keys", "result": duplicate_key_count},
    {"check": "missing_local_authority_code", "result": int(missing_key_counts["local_authority_code"])},
    {"check": "missing_local_authority_name", "result": int(missing_key_counts["local_authority_name"])},
    {"check": "missing_year", "result": int(missing_key_counts["year"])},
])

quality_summary


,check,result
0,row_count,9222
1,column_count,9
2,duplicate_local_authority_year_keys,0
3,missing_local_authority_code,0
4,missing_local_authority_name,0
5,missing_year,0


In [8]:
missing_profile = (
    cleaned
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
    .rename(columns={"index": "column_name", 0: "missing_pct"})
)

missing_profile


,column_name,missing_pct
0,country_region_code,0.00
1,country_region_name,0.00
2,local_authority_code,0.00
3,local_authority_name,0.00
4,year,0.00
5,lower_quartile_house_price,0.00
6,lower_quartile_annual_earnings,1.81
7,lower_quartile_affordability_ratio,1.81
8,median_affordability_ratio,1.84


## Add analytical fields


In [9]:
def assign_pressure_band(ratio):
    if pd.isna(ratio):
        return np.nan
    if ratio < 5:
        return "Lower pressure"
    if ratio < 8:
        return "Moderate pressure"
    if ratio < 12:
        return "High pressure"
    return "Severe pressure"


cleaned["affordability_pressure_band"] = cleaned["lower_quartile_affordability_ratio"].apply(assign_pressure_band)

latest_year = int(cleaned["year"].max())
latest = cleaned.loc[cleaned["year"] == latest_year].copy()
latest["latest_year_rank_most_pressured"] = latest["lower_quartile_affordability_ratio"].rank(
    method="dense",
    ascending=False,
)

cleaned = cleaned.merge(
    latest[["local_authority_code", "latest_year_rank_most_pressured"]],
    on="local_authority_code",
    how="left",
)

print(f"Latest year in dataset: {latest_year}")
cleaned.head()


Latest year in dataset: 2025


,country_region_code,country_region_name,local_authority_code,local_authority_name,year,lower_quartile_house_price,lower_quartile_annual_earnings,lower_quartile_affordability_ratio,median_affordability_ratio,affordability_pressure_band,latest_year_rank_most_pressured
0,E12000008,South East,E07000223,Adur,1997,49950,11662,4.28,3.54,Lower pressure,77.0
1,E12000008,South East,E07000223,Adur,1998,56500,13091,4.32,3.68,Lower pressure,77.0
2,E12000008,South East,E07000223,Adur,1999,63000,14091,4.47,3.87,Lower pressure,77.0
3,E12000008,South East,E07000223,Adur,2000,74375,14496,5.13,4.69,Moderate pressure,77.0
4,E12000008,South East,E07000223,Adur,2001,86000,11774,7.3,6.63,Moderate pressure,77.0


## Quick review of latest-year affordability pressure


In [10]:
latest_review_columns = [
    "country_region_name",
    "local_authority_name",
    "lower_quartile_house_price",
    "lower_quartile_annual_earnings",
    "lower_quartile_affordability_ratio",
    "median_affordability_ratio",
    "affordability_pressure_band",
]

latest_most_pressured = (
    cleaned.loc[cleaned["year"] == latest_year, latest_review_columns]
    .dropna(subset=["lower_quartile_affordability_ratio"])
    .sort_values("lower_quartile_affordability_ratio", ascending=False)
    .head(10)
)

latest_most_pressured


,country_region_name,local_authority_name,lower_quartile_house_price,lower_quartile_annual_earnings,lower_quartile_affordability_ratio,median_affordability_ratio,affordability_pressure_band
4059,London,Kensington and Chelsea,700000,33510,20.89,25.22,Severe pressure
6176,London,Richmond upon Thames,491875,30765,15.99,17.55,Severe pressure
7713,South East,Tandridge,387625,25167,15.4,13.23,Severe pressure
2696,South East,Elmbridge,450000,29535,15.24,15.74,Severe pressure
8757,London,Westminster,575000,37787,15.22,16.04,Severe pressure
3421,London,Haringey,440000,29021,15.16,13.44,Severe pressure
8351,London,Wandsworth,453625,31275,14.5,14.7,Severe pressure
6582,South East,Sevenoaks,360000,25237,14.26,13.43,Severe pressure
3363,London,Hammersmith and Fulham,515000,36244,14.21,13.88,Severe pressure
6002,London,Redbridge,375000,26422,14.19,14.16,Severe pressure


In [11]:
latest_least_pressured = (
    cleaned.loc[cleaned["year"] == latest_year, latest_review_columns]
    .dropna(subset=["lower_quartile_affordability_ratio"])
    .sort_values("lower_quartile_affordability_ratio", ascending=True)
    .head(10)
)

latest_least_pressured


,country_region_name,local_authority_name,lower_quartile_house_price,lower_quartile_annual_earnings,lower_quartile_affordability_ratio,median_affordability_ratio,affordability_pressure_band
1942,North East,County Durham,92500,27003,3.43,4.34,Lower pressure
1159,North West,Burnley,85000,24417,3.48,4.18,Lower pressure
3885,North West,Hyndburn,92500,26327,3.51,4.08,Lower pressure
3537,North East,Hartlepool,100000,26716,3.74,4.8,Lower pressure
4871,North East,Middlesbrough,105000,27262,3.85,4.51,Lower pressure
5799,North West,Pendle,100000,25497,3.92,4.52,Lower pressure
550,North West,Blackpool,105000,26224,4.0,4.18,Lower pressure
4117,Yorkshire and The Humber,"Kingston upon Hull, City of",110000,26613,4.13,4.12,Lower pressure
2058,North West,Cumberland,122500,29684,4.13,4.15,Lower pressure
7394,West Midlands,Stoke-on-Trent,115000,27711,4.15,4.44,Lower pressure


## Save cleaned outputs


In [12]:
cleaned_output = CLEANED_DIR / "affordability_area_year_cleaned.csv"
quality_output = CLEANED_DIR / "cleaning_quality_summary.csv"
missing_output = CLEANED_DIR / "cleaned_missing_profile.csv"

cleaned.to_csv(cleaned_output, index=False)
quality_summary.to_csv(quality_output, index=False)
missing_profile.to_csv(missing_output, index=False)

print(f"Saved cleaned data to: {cleaned_output.relative_to(PROJECT_ROOT)}")
print(f"Saved quality summary to: {quality_output.relative_to(PROJECT_ROOT)}")
print(f"Saved missing profile to: {missing_output.relative_to(PROJECT_ROOT)}")


Saved cleaned data to: data\cleaned\affordability_area_year_cleaned.csv
Saved quality summary to: data\cleaned\cleaning_quality_summary.csv
Saved missing profile to: data\cleaned\cleaned_missing_profile.csv


In [13]:
data_dictionary = pd.DataFrame([
    {"column_name": "country_region_code", "description": "ONS code for the country or English region", "type": "string"},
    {"column_name": "country_region_name", "description": "Country or English region name", "type": "string"},
    {"column_name": "local_authority_code", "description": "ONS local authority code", "type": "string"},
    {"column_name": "local_authority_name", "description": "Local authority name", "type": "string"},
    {"column_name": "year", "description": "Reference year", "type": "integer"},
    {"column_name": "lower_quartile_house_price", "description": "Lower-quartile house price from ONS affordability workbook", "type": "numeric"},
    {"column_name": "lower_quartile_annual_earnings", "description": "Lower-quartile gross annual workplace-based earnings", "type": "numeric"},
    {"column_name": "lower_quartile_affordability_ratio", "description": "Lower-quartile house price divided by lower-quartile annual earnings", "type": "numeric"},
    {"column_name": "median_affordability_ratio", "description": "Median house price divided by median annual earnings, included as a comparison measure", "type": "numeric"},
    {"column_name": "affordability_pressure_band", "description": "Banding derived from the lower-quartile affordability ratio", "type": "string"},
    {"column_name": "latest_year_rank_most_pressured", "description": "Latest-year dense rank, with 1 assigned to the highest affordability pressure", "type": "numeric"},
])

dictionary_output = DICTIONARY_DIR / "data_dictionary.csv"
data_dictionary.to_csv(dictionary_output, index=False)

data_dictionary


,column_name,description,type
0,country_region_code,ONS code for the country or English region,string
1,country_region_name,Country or English region name,string
2,local_authority_code,ONS local authority code,string
3,local_authority_name,Local authority name,string
4,year,Reference year,integer
5,lower_quartile_house_price,Lower-quartile house price from ONS affordabil...,numeric
6,lower_quartile_annual_earnings,Lower-quartile gross annual workplace-based ea...,numeric
7,lower_quartile_affordability_ratio,Lower-quartile house price divided by lower-qu...,numeric
8,median_affordability_ratio,Median house price divided by median annual ea...,numeric
9,affordability_pressure_band,Banding derived from the lower-quartile afford...,string


## Cleaning summary

The cleaned dataset is ready for the SQL analysis stage once the row counts, missing profile and latest-year rankings have been reviewed.
